In [1]:
#Array packages
import pandas as pd
import numpy as np
import xarray as xr
import netCDF4 as nc4

from scipy.stats import kendalltau
import pymannkendall as mk

#plots
import matplotlib.pyplot as plt
import rioxarray as rio
import geopandas as gpd
from shapely.geometry import mapping

#Progress meter
from dask.diagnostics import ProgressBar
from tqdm import tqdm

# Directories
import os
import glob
import dask
#import h5netcdf
import scipy

import os
#os.chdir(r"E:\academy\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity")
os.chdir(r"G:\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity")

print(os.getcwd())

G:\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity


## IMD_gridded

### 1. Merging IMD raw nc files to a single nc

In [3]:
fl_list=sorted(glob.glob('3_Data/Data_r/4_IMD gridded/*.nc'))
fl_list = [os.path.normpath(f) for f in fl_list]
ds = xr.open_mfdataset(fl_list, combine='by_coords',engine='netcdf4', parallel=True,chunks={'time': 'auto'})
ds1 = ds.rename({'LONGITUDE': 'lon', 'LATITUDE': 'lat', 'TIME': 'time'})
ds1.to_netcdf('3_Data/Data_p/4_PPT IMD/IMD_ppt_0.25.nc')

### 2. Extracting the grids corresponding to gauge station

In [ ]:
ppt_gridded=xr.open_dataset("3_Data/Data_p/4_PPT IMD/IMD_ppt_0.25.nc")
gauge_info=pd.read_csv(r"3_Data/Data_p/2_Station/1_Streamflow_data/gauge_info_p.csv")



time1=ppt_gridded.time.values
Stations=gauge_info['Station'].to_numpy()

ppt_data = np.full((len(time1), len(Stations)),np.nan)
sf_data = np.full((len(time1), len(Stations)),np.nan)

ds=xr.Dataset(
    {"Ppt" : (['time','Station'],ppt_data),
     "Streamflow" : (['time','Station'],sf_data)
    },
    coords={
        "time":time1,
        "Station":Stations}
    )

lon_grid=ppt_gridded.lon.values
lat_grid=ppt_gridded.lat.values

for s,stn in enumerate(gauge_info['Station'].values):

    lat_lon=gauge_info.loc[s,['Latitude','Longitude']]

    #Taking nearest longitude and latitude from gridded ppt
    lat_station=lat_grid[abs(lat_grid-lat_lon[0]).argmin()]
    lon_station=lon_grid[abs(lon_grid-lat_lon[1]).argmin()]


    ds['Ppt'].loc[{'Station':stn,'time':time1}]=ppt_gridded['RAINFALL'].sel(lat=lat_station,lon=lon_station).values

    data1=pd.read_csv(f"3_Data/Data_p/2_Station/1_Streamflow_data/{gauge_info['GaugeID'][s]}.csv",index_col=0,parse_dates=True)
    data1=data1.resample('D').asfreq()
    data1=data1.reindex(time1)

    ds['Streamflow'].loc[{'Station':stn,'time':time1}]=data1['Streamflow (cumecs)'].values
    
ds.to_netcdf('3_Data/Data_p/4_PPT IMD/PPT_station.nc')

In [116]:
ds.to_netcdf('3_Data/Data_p/4_PPT IMD/PPT_station.nc')